Optimisation de l'entrainement pour `focus` 
This is the same function as used in `10_Transfer_learning_what_networks.ipynb`
> ... TODO ... # TODO test without circular padding, with Adam, with no warmstart 

    model = torchvision.models.resnet18(weights=None)

In [1]:
from retinotopy import *

Running on GPU :  NVIDIA RTX A2000 12GB #GPU= 1


In [2]:
args = Params()
data_set_type = 'bbox' # Select your root between : 'boxes', 'focus', 'full'
print(f'{data_set_type=}')
args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images to perform the training

data_set_type='bbox'


# optimize meta-parameters

In [3]:
# print(path_save)
# %ls -l {path}*
# %rm {path} + '.sqlite3'

In [4]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [5]:
study_name = datetag + '_optuna'

#  TODO study_name = datetag + '_optuna-Adam'

model_name = 'resnet101'
do_polar = True

model_filename = get_filename(data_cache, datetag, data_set_type, model_name, do_polar) + '.pt'


scan_dicts= {'lr' : [0.00001], 'beta2' : [.1], 'momentum' : [0.09]   }

label_dicts= { 'lr' : 'lr',  'beta2' : 'Adam beta2', 'momentum' : 'momentum'}

In [6]:
model_filename

'cached_data/2025-01-05_bbox_resnet101_retino.pt'

In [7]:
%ls -l {model_filename}

ls: impossible d'accéder à 'cached_data/2025-01-05_bbox_resnet101_retino.pt': Aucun fichier ou dossier de ce nom


In [8]:
subplotpars_scan = SubplotParams(left=0.125, right=.95, bottom=0.25, top=.975)
max_threshold = .999
for key in scan_dicts:
    filename = f'{data_cache}/{study_name}_{key}.json'

        
    if os.path.isfile(filename):
        df_scan = pd.read_json(filename)
    else:
        print(50*'=')
        print('Scanning along', key, "=", label_dicts[key])
        print(50*'=')
        
        measure_columns = [key, 'accuracy']
        df_scan = pd.DataFrame([], columns=measure_columns)
        i_loc = 0
        for i_value, value in enumerate(scan_dicts[key]):
            print('i_value', i_value + 1, ' /', len(scan_dicts[key]), key, '=', value)

            opt =  Params()

            new_dict = asdict(opt)
            new_dict[key] = value
            new_opt = Params(**new_dict)
            new_opt.n_train_stop = 100000
            new_opt.num_epochs = 1
            new_opt.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images to perform the training
            
            def objective(trial):
                # new_opt.rs_min = trial.suggest_float('rs_min', -1, 1.)
                # new_opt.rs_max = trial.suggest_float('rs_max', -7, -4)
                scale = 10
                new_opt.momentum = trial.suggest_float('momentum', opt.momentum/scale, min(opt.momentum*scale, max_threshold), log=True)
                if new_opt.beta2>0: new_opt.beta2 = trial.suggest_float('beta2', new_opt.beta2/scale, min(new_opt.beta2*scale, max_threshold), log=True)
                scale = 50
                new_opt.lr = trial.suggest_float('lr', opt.lr / scale, opt.lr * scale, log=True)

                # get the architecture of the network
                model_retrain = load_model(model_name=model_name, model_path=model_filename, do_scratch=new_opt.do_scratch, do_circular=new_opt.do_polar, verbose=False).to(device)
                                
                # load the data
                dataloaders = datasets_transforms(new_opt, verbose=False)

                # train and get accuracy on the validation set
                _, df_train = train_model(new_opt, model_retrain, dataloaders=dataloaders, verbose=False)
                accuracy = df_train['avg_acc_val'].mean()
                
                return accuracy

            opt_tuna= dict(storage=f"sqlite:///{os.path.join(data_cache, study_name)}.sqlite3", direction='maximize', load_if_exists=True,study_name=f"{key} = {value}")

            # 3. Create a study object and optimize the objective function.
            study = optuna.create_study(**opt_tuna)
            study.optimize(objective, n_trials=150, n_jobs=1, show_progress_bar=True)
            print(50*'-.')
            print("Best params: ", study.best_params)
            print("Best value: ", study.best_value)
            print("Best Trial: ", study.best_trial)
            print("Trials: ", study.trials)
            print(50*'-.')
            df_scan.loc[i_loc] = {key:value, 'accuracy':study.best_value}
            i_loc += 1
        df_scan.to_json(filename, orient='index', indent=2)
    print(df_scan)
    print(50*'=')

    fig, ax = plt.subplots(figsize=(fig_width, fig_width/phi), subplotpars=subplotpars_scan)
    gp_scan = df_scan[[key, 'accuracy']].groupby([key])
    means = gp_scan.mean()
    errors = gp_scan.std()
    means.plot.bar(yerr=errors, ax=ax, capsize=4, rot=-60, legend=False, color='r', alpha=.5)
    
    ax.set_ylabel('Accuracy')
    ax.set_xlabel(key + ' = ' +label_dicts[key])
    #ax.set_xscale('log')

    ax.set_ylim(0, 1)
    #fig = ax.get_figure()
    # pos = ax.get_position()
    # print(pos)
    plt.show()

Scanning along lr = lr
i_value 1  / 1 lr = 1e-05


  0%|          | 0/150 [00:00<?, ?it/s]

Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to /home/laurent/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth
100%|██████████| 171M/171M [00:03<00:00, 52.3MB/s]


[W 2025-01-14 14:44:44,029] Trial 0 failed with parameters: {'momentum': 0.4855718413710951, 'lr': 2.2871270277350578e-05} because of the following error: FileNotFoundError(2, 'No such file or directory').
Traceback (most recent call last):
  File "/home/laurent/sdrive_cnrs/hot_from_git/GitHub/Retinotopy/.venv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_708750/1960705738.py", line 39, in objective
    model_retrain = load_model(model_name=model_name, model_path=model_filename, do_scratch=new_opt.do_scratch, do_circular=new_opt.do_polar, verbose=False).to(device)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/laurent/sdrive_cnrs/hot_from_git/GitHub/Retinotopy/retinotopy.py", line 740, in load_model
    model.load_state_dict(torch.load(mode

FileNotFoundError: [Errno 2] No such file or directory: 'cached_data/2025-01-05_bbox_resnet101_retino.pt'